# Notebook 5 — M3: NLP Orchestration Layer

**Real Estate Intelligence Platform (REIP)**

Construimos y validamos el diseño de M3: un flujo que recibe una consulta conversacional en
español, extrae criterios estructurados vía LLM, decide si la extracción es suficientemente
confiable para proceder (y si no, pide clarificación en vez de forzar una respuesta), y orquesta
M1 (Preference Matching) y M2 (Property Valuation) sobre esos criterios. Es el último notebook de
Feature 6.2 y depende de los artifacts ya cerrados de 6.2.3 (embeddings + scorer híbrido), 6.2.4
(KNN semáforo de precio) y 6.2.5 (KMeans segmentación).

Como 6.2.6, este es un notebook de tipo **Experimentación**, no de Entrenamiento: no hay una
métrica de error que minimizar, sino un diseño de mecanismo (extracción + umbral de confianza +
fallback) que reportamos con evidencia de su comportamiento, no con MAE ni Silhouette.

**Alcance:** este notebook valida el diseño de las Features 4.1 y 4.2 del WBS de Épica 4
(extracción de intención, lógica de fallback, conjunto de validación). La Feature 4.3
(endpoints `POST /search/nlp` y `/search/filtros`) es implementación de backend en producción con
FastAPI — fuera del alcance de 6.2.7. Lo que construimos aquí es la simulación del contrato de
función que ese backend consumirá.

## Verificación de entorno y rutas (§7.4)

Confirmamos la convención ya establecida en 6.2.3/6.2.6: `.env` vive en la raíz del repo, no en
`notebooks/`, y el kernel corre con `cwd = notebooks/`, así que usamos `find_dotenv(usecwd=True)`
en vez de `load_dotenv()` simple.

**Dependencias de este notebook (corregidas en `Feature_6.2_Contexto_Ejecucion_v2.md` respecto a
la versión original del WBS):** a diferencia de 6.2.6, M3 no depende únicamente de
`GEMINI_API_KEY` — orquesta M1 y M2, así que también depende de sus artifacts ya cerrados:

- `pipeline/models/embeddings_catalogo_6_2_3_raw.pkl` — M1, embeddings del catálogo (6.2.3).
- `pipeline/models/knn_semaforo_precio_6_2_4.pkl` + `escalador_knn_6_2_4.pkl` — M2, semáforo de
  precio (6.2.4). El artifact es un `dict` con las columnas exactas del modelo entrenado
  (`columnas`), no solo el objeto sklearn — verificamos esto contra el archivo real, no lo
  asumimos, porque esas columnas son precisamente lo que determina qué zonas puede cubrir el
  modelo.
- `pipeline/models/kmeans_segmentacion_6_2_5.pkl` + `escalador_kmeans_6_2_5.pkl` — M2, segmentación
  de mercado (6.2.5).
- `pipeline/models/quality_scores_6_2_6_raw.pkl` — M2, Quality Scorer (6.2.6), disponible solo
  para las 150 propiedades de la muestra evaluada, no el catálogo completo.
- `pipeline/data/processed/catalogo_residencial_limpio_6_2_1.csv` — catálogo base (6.2.1), usado
  por M1 para el filtro estructurado sobre criterios extraídos.

Todas las rutas se confirman contra el repo real en la celda siguiente antes de usarlas.

In [1]:
import sys
import os
import json
import time
import pickle
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

RUTA_CATALOGO = REPO_ROOT / "pipeline" / "data" / "processed" / "catalogo_residencial_limpio_6_2_1.csv"
RUTA_MODELOS = REPO_ROOT / "pipeline" / "models"

RUTA_KNN = RUTA_MODELOS / "knn_semaforo_precio_6_2_4.pkl"
RUTA_ESCALADOR_KNN = RUTA_MODELOS / "escalador_knn_6_2_4.pkl"
RUTA_KMEANS = RUTA_MODELOS / "kmeans_segmentacion_6_2_5.pkl"
RUTA_ESCALADOR_KMEANS = RUTA_MODELOS / "escalador_kmeans_6_2_5.pkl"
RUTA_EMBEDDINGS_CATALOGO = RUTA_MODELOS / "embeddings_catalogo_6_2_3_raw.pkl"
RUTA_QUALITY_SCORES = RUTA_MODELOS / "quality_scores_6_2_6_raw.pkl"

for ruta in [RUTA_CATALOGO, RUTA_KNN, RUTA_ESCALADOR_KNN, RUTA_KMEANS, RUTA_ESCALADOR_KMEANS,
             RUTA_EMBEDDINGS_CATALOGO, RUTA_QUALITY_SCORES]:
    assert ruta.exists(), f"No existe {ruta}"
    print(f"OK  {ruta.relative_to(REPO_ROOT)}")

RUTA_MODELOS.mkdir(exist_ok=True)


def cargar_checkpoint(ruta):
    if ruta.exists():
        with open(ruta, "rb") as f:
            return pickle.load(f)
    return {}


def guardar_checkpoint(ruta, obj):
    with open(ruta, "wb") as f:
        pickle.dump(obj, f)


OK  pipeline/data/processed/catalogo_residencial_limpio_6_2_1.csv
OK  pipeline/models/knn_semaforo_precio_6_2_4.pkl
OK  pipeline/models/escalador_knn_6_2_4.pkl
OK  pipeline/models/kmeans_segmentacion_6_2_5.pkl
OK  pipeline/models/escalador_kmeans_6_2_5.pkl
OK  pipeline/models/embeddings_catalogo_6_2_3_raw.pkl
OK  pipeline/models/quality_scores_6_2_6_raw.pkl


## Confirmación del modelo LLM contra la API real

Siguiendo la misma regla ya aplicada en 6.2.3 y 6.2.6 (`client.models.list()` no es fuente
confiable de disponibilidad real — solo una llamada de prueba lo confirma), verificamos de nuevo
el estado de `gemini-3.5-flash` antes de heredar la decisión de 6.2.6, que había cambiado a
`gemini-3-flash-preview` por una ventana de indisponibilidad sostenida (100% de fallo en un lote
diagnóstico completo).

**No heredamos esa decisión sin comprobarla de nuevo.** Un primer sondeo rápido (10 llamadas) dio
50% de éxito — ambiguo entre "saturación puntual" y "degradación sostenida". Corrimos el mismo
tamaño de lote diagnóstico que estableció el criterio en 6.2.6 (25 llamadas, timeout explícito de
45s, backoff corto de ~20s ante 503/504) antes de decidir.

In [2]:
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv(usecwd=True))

from google import genai
from google.genai import types
from google.genai.errors import ServerError

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(timeout=45_000),
)
# Sin este timeout explicito (ms), el cliente hereda timeout=None de httpx -- convencion
# establecida en 6.2.6 tras un bloqueo real de ~3 horas por una conexion estancada.

RUTA_DIAG_MODELO = RUTA_MODELOS / "model_availability_check_6_2_7_batch25.pkl"

diag = cargar_checkpoint(RUTA_DIAG_MODELO)
if not diag:
    resultados_diag = []
    for i in range(25):
        for attempt in range(2):
            try:
                t0 = time.time()
                resp = client.models.generate_content(
                    model="gemini-3.5-flash", contents=f"Responde solo con el numero {i}."
                )
                resultados_diag.append({"i": i, "ok": True, "dt": round(time.time() - t0, 1),
                                         "attempts": attempt + 1})
                break
            except ServerError:
                if attempt == 0:
                    time.sleep(20)
                else:
                    resultados_diag.append({"i": i, "ok": False, "attempts": attempt + 1})
    diag = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "model": "gemini-3.5-flash",
            "results": resultados_diag}
    guardar_checkpoint(RUTA_DIAG_MODELO, diag)

n = len(diag["results"])
ok = sum(1 for r in diag["results"] if r["ok"])
reintentos = sum(1 for r in diag["results"] if r["ok"] and r.get("attempts", 1) == 2)
print(f"gemini-3.5-flash -- exito: {ok}/{n} ({ok/n*100:.0f}%) | requirieron reintento (503 puntual): {reintentos}/{n}")


gemini-3.5-flash -- exito: 25/25 (100%) | requirieron reintento (503 puntual): 2/25


**Resultado:** 25/25 = 100% de éxito, con solo 2/25 (8%) requiriendo el reintento corto por
un 503 puntual — comportamiento consistente con saturación momentánea absorbida por el backoff ya
establecido, no con la degradación sostenida que forzó el cambio en 6.2.6. Esto es una mejora real
respecto al hallazgo de 6.2.6, no una repetición del mismo — la ventana de indisponibilidad total
ya no está presente al momento de esta corrida.

**Decisión: usamos `gemini-3.5-flash`** (identificador GA, fijo) en vez de heredar
`gemini-3-flash-preview` de 6.2.6. Recupera el argumento de reproducibilidad/citación para el
paper que el modelo "preview" no ofrece — con la advertencia de que la disponibilidad de modelos
Gemini es una condición del momento de ejecución, no una garantía permanente; cualquier corrida
futura debería repetir esta misma verificación antes de asumir el resultado vigente.

## Contrato de función v0 — decisión de arquitectura

M3 no llama `pickle.load()` + la función de M1/M2 directo en aislamiento. Definimos un wrapper de
input/output fijo que imita la interfaz que tendrá el backend real de Épica 4 (FastAPI llamando a
M1/M2) — probamos el patrón de orquestación, no solo el resultado del modelo combinado. Se marca
explícitamente como **`v0 — sujeto a revisión en Épica 4`** y se define en una sola celda
reutilizable, no reescrito en cada celda de uso.

**Regla de cobertura estructural en M2 (KNN):** el artifact `knn_semaforo_precio_6_2_4.pkl` guarda
la lista exacta de columnas (`columnas`) del modelo ya entrenado — confirmamos que **no incluye**
`zona_Pedregal` ni `zona_Parque Lefevre` (excluidas del pool de entrenamiento por volumen
insuficiente, Acta 1.2 §5.3) ni admite `tipo_inmueble` distinto de Apartamento (6.2.4 restringió el
pool a Apartamentos por contaminación no residencial). Esto no es "pocos comparables" — es una
imposibilidad estructural de construir el vector de features que el modelo espera. El contrato
detecta esto **antes** de llamar `.predict()` y devuelve `cobertura_insuficiente=True` de forma
explícita, en vez de forzar una predicción fuera del dominio de entrenamiento del modelo o dejar
que falle con un error no manejado — mismo principio que la UI de "cobertura de datos insuficiente"
ya usada para Pedregal y Costa del Este en el Zone Health Index (decisiones cerradas, CLAUDE.md).

**Regla de honestidad en la respuesta final:** la respuesta ensamblada distingue explícitamente
cuatro combinaciones (con/sin candidatos de M1 × con/sin cobertura de M2), no una sola rama
genérica — un primer borrador de esta función, corregido durante la verificación puntual más
abajo, colapsaba "sin candidatos" y "con candidatos" en el mismo mensaje cuando M2 no tenía
cobertura, lo que producía una respuesta engañosa ("te mostramos los resultados... " cuando no
había ninguno).

In [3]:
CONTRATO_VERSION = "v0"  # sujeto a revision en Epica 4

df_catalogo = pd.read_csv(RUTA_CATALOGO)

with open(RUTA_KNN, "rb") as f:
    knn_artifact = pickle.load(f)
with open(RUTA_ESCALADOR_KNN, "rb") as f:
    escalador_knn_artifact = pickle.load(f)

TIPO_SINGULAR_A_PLURAL = {
    "Apartamento": "Apartamentos", "Casa": "Casas", "Edificio": "Edificios",
    "Local": "Locales", "Terreno": "Terrenos",
}


def m1_buscar_matches(criterios: dict, top_k: int = 5) -> dict:
    # Contrato v0 - M1 (Preference Matching): filtro estructurado sobre el catalogo real.
    # Una busqueda puntual de M3 no trae un perfil de lifestyle (el insumo del componente
    # semantico de 6.2.3), asi que este contrato usa el filtro estructurado -- el mismo
    # insumo que compone el 0.6 del scorer hibrido ya validado con Precision@k en 6.2.3.
    df = df_catalogo.copy()
    df = df[df["corregimiento"] != "zona_no_determinada"]
    df = df[df["precio_no_evaluable"] == False]

    if criterios.get("zona"):
        df = df[df["corregimiento"] == criterios["zona"]]
    if criterios.get("tipo_inmueble"):
        tipo_plural = TIPO_SINGULAR_A_PLURAL.get(criterios["tipo_inmueble"])
        df = df[df["tipo_inmueble"] == tipo_plural]
    if criterios.get("precio_min"):
        df = df[df["price_usd"] >= criterios["precio_min"]]
    if criterios.get("precio_max"):
        df = df[df["price_usd"] <= criterios["precio_max"]]
    if criterios.get("habitaciones_min"):
        df = df[df["bedrooms"] >= criterios["habitaciones_min"]]
    if criterios.get("banos_min"):
        df = df[df["bathrooms"] >= criterios["banos_min"]]

    return {
        "contrato_version": CONTRATO_VERSION,
        "n_total_filtrado": len(df),
        "candidatos": df.head(top_k).to_dict("records"),
    }


def m2_enriquecer_semaforo(candidatos: list, zona: str, tipo_inmueble: str) -> dict:
    # Contrato v0 - M2 (KNN semaforo de precio). Chequeo estructural PREVIO a llamar el
    # modelo: si la zona no tiene columna dummy en el set de entrenamiento o el
    # tipo_inmueble no es Apartamento, no se intenta una prediccion fuera del dominio.
    columnas_modelo = knn_artifact["columnas"]
    zona_col = f"zona_{zona}" if zona else None
    cobertura_ok = (zona_col in columnas_modelo) and (tipo_inmueble == "Apartamento")

    if not cobertura_ok:
        motivo = []
        if zona_col not in columnas_modelo:
            motivo.append(f"zona '{zona}' excluida del pool de entrenamiento KNN/RF (Acta 1.2 par. 5.3)")
        if tipo_inmueble != "Apartamento":
            motivo.append(f"tipo_inmueble '{tipo_inmueble}' fuera del pool de entrenamiento (6.2.4 lo restringio a Apartamentos)")
        return {"contrato_version": CONTRATO_VERSION, "cobertura_insuficiente": True,
                "motivo": "; ".join(motivo), "candidatos_enriquecidos": []}

    modelo = knn_artifact["modelo"]
    escalador = escalador_knn_artifact["escalador"]
    columnas_numericas = escalador_knn_artifact["columnas_numericas"]
    umbral = knn_artifact["umbral_semaforo"]

    enriquecidos = []
    for c in candidatos:
        fila = pd.DataFrame([{col: 0 for col in columnas_modelo}])
        fila.loc[0, "bedrooms"] = c["bedrooms"]
        fila.loc[0, "bathrooms"] = c["bathrooms"]
        fila.loc[0, "area_m2"] = c["area_m2"]
        if zona_col in fila.columns:
            fila.loc[0, zona_col] = 1
        fila[columnas_numericas] = escalador.transform(fila[columnas_numericas])
        pred = modelo.predict(fila[columnas_modelo])[0]
        residual_rel = (c["price_usd"] - pred) / pred
        if abs(residual_rel) <= umbral * 1.5:
            semaforo = "amarillo"
        elif residual_rel > umbral * 1.5:
            semaforo = "rojo (sobrevalorado)"
        else:
            semaforo = "verde (subvalorado)"
        enriquecidos.append({**c, "precio_predicho_knn": round(pred, 2), "semaforo": semaforo})

    return {"contrato_version": CONTRATO_VERSION, "cobertura_insuficiente": False,
            "candidatos_enriquecidos": enriquecidos}


def orquestar_m3(criterios: dict) -> dict:
    # Contrato v0 - orquestacion completa M3: M1 -> M2, ensamblado final honesto sobre
    # las 4 combinaciones posibles (con/sin candidatos x con/sin cobertura de M2).
    resultado_m1 = m1_buscar_matches(criterios)
    resultado_m2 = m2_enriquecer_semaforo(
        resultado_m1["candidatos"], criterios.get("zona"), criterios.get("tipo_inmueble")
    )

    n_matches = resultado_m1["n_total_filtrado"]
    sin_matches = n_matches == 0

    if sin_matches and resultado_m2["cobertura_insuficiente"]:
        respuesta = (
            f"No encontramos propiedades que coincidan con esos criterios en {criterios.get('zona')} "
            f"(0 candidatos). Nota adicional: aunque hubiera candidatos, tampoco podriamos mostrar el "
            f"semaforo de precio para esta zona/tipo, porque el modelo de valoracion no tiene "
            f"suficientes datos de entrenamiento ahi ({resultado_m2['motivo']})."
        )
    elif sin_matches:
        respuesta = (
            f"No encontramos propiedades que coincidan con esos criterios en "
            f"{criterios.get('zona', 'la zona indicada')} (0 candidatos). Prueba ajustando el rango "
            f"de precio o el numero de habitaciones."
        )
    elif resultado_m2["cobertura_insuficiente"]:
        respuesta = (
            f"Encontramos {n_matches} propiedades que coinciden con tu busqueda en "
            f"{criterios.get('zona')}. No podemos mostrar el semaforo de precio para esta zona/tipo "
            f"porque el modelo de valoracion no tiene suficientes datos de entrenamiento ahi "
            f"({resultado_m2['motivo']}) - te mostramos los resultados de coincidencia sin ese dato."
        )
    else:
        respuesta = (
            f"Encontramos {n_matches} propiedades que coinciden con tu busqueda, "
            f"con semaforo de precio incluido."
        )

    return {"contrato_version": CONTRATO_VERSION, "m1": resultado_m1, "m2": resultado_m2,
            "respuesta_final": respuesta}


## Fase 1 — Calibración del umbral de confianza (WBS 4.1.1, 4.1.3)

Definimos el esquema de extracción (con un campo `confianza` explícito) y el system prompt del
extractor, y los probamos contra un conjunto de 10 consultas de calibración — 5 claras y 5
ambiguas, más un caso de control adicional fuera del mínimo del WBS (un caso límite,
deliberadamente mixto: entidades duras junto a un calificativo sin ancla numérica en la misma
frase) para ubicar mejor el punto de corte antes de fijar un número.

**Advertencia de validación circular, mismo patrón que 6.2.3:** estas 11 consultas fueron
diseñadas por nosotros mismos, no recolectadas de usuarios reales — evalúan si el mecanismo hace
lo que se diseñó para hacer (distinguir estructura clara de vaguedad), no cuál sería la
distribución real de confianza en tráfico de producción.

In [4]:
from pydantic import BaseModel, Field
from typing import Optional, List

ZONAS_SCOPE = [
    "San Francisco", "Bella Vista", "Parque Lefevre", "Betania", "Pedregal",
    "El Cangrejo", "Marbella", "Obarrio", "Costa del Este",
]


class ExtraccionIntencion(BaseModel):
    es_consulta_inmobiliaria: bool = Field(description="False si la consulta no trata sobre busqueda/compra de propiedades residenciales, sin importar que mencione una zona valida")
    zona: Optional[str] = Field(description="Una de las 9 zonas del scope, o null si no se menciona o no mapea a ninguna")
    zona_mencion_texto: Optional[str] = Field(description="Texto literal de CUALQUIER ubicacion mencionada, incluso si no esta en la lista de 9 zonas validas. Null solo si no se menciono ninguna ubicacion.")
    tipo_inmueble: Optional[str] = Field(description="Apartamento, Casa, Edificio, Local, Terreno, o null si no se menciona")
    precio_min: Optional[float] = Field(description="Precio minimo en USD si se menciona un rango o piso explicito, si no null")
    precio_max: Optional[float] = Field(description="Precio maximo/techo en USD si se menciona explicitamente un numero, si no null")
    habitaciones_min: Optional[int] = Field(description="Numero minimo de habitaciones si se menciona explicitamente, si no null")
    banos_min: Optional[int] = Field(description="Numero minimo de banos si se menciona explicitamente, si no null")
    caracteristicas_cualitativas: List[str] = Field(description="Frases cualitativas sin ancla numerica ni categorica clara")
    confianza: float = Field(description="0.0 a 1.0: que tan completa y no-ambigua es la extraccion de entidades estructuradas")
    razon_confianza: str = Field(description="Explicacion breve de por que se asigno ese nivel de confianza")


SYSTEM_PROMPT = f'''Eres el modulo de extraccion de intencion de un buscador conversacional de propiedades \
residenciales en Panama (REIP). Tu tarea es extraer entidades estructuradas de la consulta del usuario \
en espanol, y asignar un puntaje de confianza a la extraccion.

PRIMER CHEQUEO -- es_consulta_inmobiliaria: evalua si la consulta trata sobre buscar/comprar una \
propiedad residencial. Si el usuario pregunta por otra cosa (un servicio, un producto, informacion \
general) aunque mencione una zona valida de pasada, marca es_consulta_inmobiliaria=False.

Zonas validas del scope (unicas que puedes asignar al campo zona): {", ".join(ZONAS_SCOPE)}.
Si el usuario menciona una ubicacion que NO esta en esta lista, o una descripcion vaga de zona \
("zona buena", "cerca del trabajo") que no mapea directamente a una de estas 9, deja el campo zona \
en null PERO copia el texto literal de esa mencion en zona_mencion_texto. Si el usuario no \
menciono ninguna ubicacion, zona_mencion_texto tambien debe ser null. Si menciona dos zonas \
validas sin criterio de desempate, zona tambien queda null, y zona_mencion_texto debe contener \
ambas menciones.

Tipos de inmueble validos: Apartamento, Casa, Edificio, Local, Terreno.

Reglas de confianza (aplica solo cuando es_consulta_inmobiliaria=True):
- ALTA (>0.7): la mayoria de las entidades mencionadas tienen ancla numerica o categorica clara.
  Campos simplemente ausentes NO bajan la confianza -- ausencia no es ambiguedad.
- BAJA (<0.4): la consulta usa calificativos subjetivos sin ancla en lugar de numeros, o hay
  conflicto real (dos zonas sin desempate).
- MEDIA: mezcla de entidades ancladas y calificativos sueltos en la misma consulta.

Extrae SOLO lo que esta explicito o fuertemente implicito. No inventes numeros para calificativos vagos.'''


def extraer(consulta: str) -> dict:
    for attempt in range(2):
        try:
            resp = client.models.generate_content(
                model="gemini-3.5-flash",
                contents=consulta,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    response_mime_type="application/json",
                    response_schema=ExtraccionIntencion,
                    thinking_config=types.ThinkingConfig(thinking_level="LOW"),
                ),
            )
            return json.loads(resp.text)
        except ServerError:
            if attempt == 0:
                time.sleep(20)
            else:
                raise
    raise RuntimeError("unreachable")


In [5]:
CONSULTAS_CALIBRACION = {
    "C1": "Busco un apartamento de 2 habitaciones en San Francisco por menos de $250,000",
    "C2": "Quiero una casa en Bella Vista, presupuesto maximo $400,000",
    "C3": "Apartamento de 3 habitaciones y 2 banos en Costa del Este",
    "C4": "Necesito algo en Obarrio entre $300,000 y $500,000",
    "C5": "Apartamento en Betania, minimo 2 banos",
    "A1": "Quiero algo bonito y comodo para vivir con mi familia",
    "A2": "Busco propiedad cerca del trabajo, no muy caro",
    "A3": "Apartamento en la zona buena, con buena plusvalia",
    "A4": "Algo entre San Francisco y Costa del Este, lo que sea mas conveniente",
    "A5": "Casa grande, varios cuartos, presupuesto flexible",
    "L1_LIMITE": "Apartamento de 2 habitaciones en El Cangrejo, precio razonable",
}

RUTA_CALIBRACION = RUTA_MODELOS / "calibracion_umbral_6_2_7_checkpoint.pkl"
resultados_calibracion = cargar_checkpoint(RUTA_CALIBRACION)

for etiqueta, consulta in CONSULTAS_CALIBRACION.items():
    if resultados_calibracion.get(etiqueta, {}).get("ok"):
        continue
    try:
        r = extraer(consulta)
        resultados_calibracion[etiqueta] = {"consulta": consulta, "extraccion": r, "ok": True}
    except Exception as e:
        resultados_calibracion[etiqueta] = {"consulta": consulta, "error": str(e), "ok": False}
    guardar_checkpoint(RUTA_CALIBRACION, resultados_calibracion)

for grupo, etiquetas in [("CLARAS", ["C1","C2","C3","C4","C5"]),
                          ("AMBIGUAS", ["A1","A2","A3","A4","A5"]),
                          ("LIMITE", ["L1_LIMITE"])]:
    vals = [resultados_calibracion[e]["extraccion"]["confianza"] for e in etiquetas]
    print(f"{grupo}: {vals} -> promedio={sum(vals)/len(vals):.3f}")


CLARAS: [0.95, 0.95, 0.95, 0.95, 0.95] -> promedio=0.950
AMBIGUAS: [0.2, 0.2, 0.35, 0.3, 0.35] -> promedio=0.280
LIMITE: [0.8] -> promedio=0.800


**Resultado:** confianza promedio 0.950 en el grupo claro, 0.280 en el grupo ambiguo, 0.800
en el caso límite. El caso límite se comporta **como claro, no como ambiguo** — el modelo no
penaliza la presencia de un calificativo subjetivo ("precio razonable") cuando coexiste con
suficientes entidades duras (zona, tipo, habitaciones); solo penaliza fuerte cuando las entidades
duras faltan o entran en conflicto. Hay un salto limpio de 0.45 entre el techo del grupo ambiguo
(0.35) y el caso límite (0.80), sin ningún punto intermedio observado en estos 11 casos.

### Umbral de confianza fijado: 0.65

**Razón:** sesgo conservador — el costo de proceder con confianza insuficiente (resultados de
búsqueda equivocados, silenciosos para el usuario) es mayor que el costo de pedir clarificación de
más (fricción menor, explícita). El valor mantiene colchón de seguridad bajo el caso límite (0.80)
sin acercarse al techo del grupo ambiguo (0.35).

**CALIBRACIÓN PROVISIONAL — no es un umbral optimizado.** 11 puntos con un salto limpio de 0.45 y
cero casos intermedios reales no prueban que 0.65 sea el punto óptimo de corte; solo confirman que
cae dentro de un rango seguro dados los datos disponibles en este ejercicio de calibración. El
punto de quiebre genuino del mecanismo (dónde exactamente el modelo empieza a fallar) permanece
sin observar directamente — requeriría más casos con confianza intermedia real, no diseñados por
nosotros con la intención expresa de caer en un extremo u otro.

## Fase 2 — Medición final: tasa de éxito, cobertura, tema y ambigüedad (WBS 4.2.2, 4.2.3)

Construimos el conjunto representativo mínimo del WBS 4.2.2 (10 válidas + 5 fuera de alcance + 5
ambiguas) — distinto del conjunto de calibración de la Fase 1, usado solo para fijar el umbral, no
para medir nada.

Diseñamos los 20 casos separando deliberadamente dos ejes de "fuera de alcance" que un solo
mecanismo de confianza no puede distinguir por sí solo (hallazgo verificado antes de correr nada):
una consulta bien formada sobre una zona real pero no cubierta por el scope (`San Miguelito`,
`Juan Díaz`) puede extraer con confianza igual de alta que una consulta válida, porque sus
entidades están igual de ancladas — el problema no es ambigüedad de redacción, es cobertura
geográfica. Lo mismo aplica a una consulta que menciona una zona válida de pasada pero no es sobre
bienes raíces (`F4`). Por eso el mecanismo final usa **tres chequeos deterministicos e
independientes**, no solo el score de confianza del LLM:

1. `es_consulta_inmobiliaria == False` → fallback por fuera de tema.
2. Ubicación mencionada (`zona_mencion_texto`) que no contiene ninguna de las 9 zonas válidas,
   con `zona` resuelta a null → fallback por cobertura geográfica (mensaje distinto: *"no
   cubrimos esa zona todavía"*, no *"no entendí tu consulta"*).
3. `confianza < 0.65` → fallback por ambigüedad.

**Corrección aplicada durante la implementación:** la primera versión del chequeo 2 disparaba
cobertura ante cualquier `zona_mencion_texto` no nulo, sin distinguir "ubicación real fuera del
scope" de "dos zonas válidas mencionadas sin desempate" (este segundo caso sí es ambigüedad, no un
problema de cobertura). Corregido verificando si el texto mencionado contiene el nombre de alguna
de las 9 zonas válidas antes de clasificarlo como cobertura — si lo contiene, cae al chequeo de
confianza en su lugar.

In [6]:
CONSULTAS_MEDICION = {
    "V1": "Apartamento de 1 habitación en El Cangrejo, menos de $150,000",
    "V2": "Casa en Betania con al menos 3 habitaciones y 2 baños",
    "V3": "Apartamento en Marbella entre $250,000 y $450,000",
    "V4": "Busco un edificio en San Francisco como inversión, presupuesto hasta $1,000,000",
    "V5": "Apartamento de 2 habitaciones en Costa del Este, mínimo 2 baños",
    "V6": "Apartamento en Parque Lefevre, menos de $300,000",
    "V7": "Apartamento en Pedregal de 3 habitaciones",
    "V8": "Apartamento en Obarrio, 1 habitación, menos de $200,000",
    "V9": "Apartamento de 3 habitaciones en Bella Vista entre $400,000 y $700,000",
    "V10": "Apartamento en San Francisco con 2 baños, presupuesto máximo $280,000",
    "F1": "Busco un apartamento en Juan Díaz",
    "F2": "Quiero una casa en San Miguelito, 3 habitaciones",
    "F3": "Busco terreno agrícola en Chiriquí para sembrar",
    "F4": "¿Cuál es el mejor taller mecánico cerca de Bella Vista?",
    "F5": "Quiero comprar un lote industrial cerca del puerto de Colón",
    "AM1": "Algo económico para empezar, no sé bien dónde",
    "AM2": "Busco algo espacioso, con buena vista, en una zona tranquila",
    "AM3": "Necesito una propiedad accesible, no importa el tamaño",
    "AM4": "Quiero invertir en algo con potencial, en Bella Vista o San Francisco, lo que rinda más",
    "AM5": "Apartamento moderno con buenos acabados, presupuesto normal",
}

UMBRAL_CONFIANZA = 0.65


def zona_texto_contiene_zona_valida(texto):
    if texto is None:
        return False
    return any(z.lower() in texto.lower() for z in ZONAS_SCOPE)


def clasificar(extraccion: dict) -> str:
    if not extraccion["es_consulta_inmobiliaria"]:
        return "fallback_fuera_tema"
    if (extraccion["zona_mencion_texto"] is not None and extraccion["zona"] is None
            and not zona_texto_contiene_zona_valida(extraccion["zona_mencion_texto"])):
        return "fallback_cobertura"
    if extraccion["confianza"] < UMBRAL_CONFIANZA:
        return "fallback_ambiguedad"
    return "exito"


RUTA_MEDICION = RUTA_MODELOS / "medicion_final_6_2_7_checkpoint.pkl"
resultados_medicion = cargar_checkpoint(RUTA_MEDICION)

for r in resultados_medicion.values():
    if r.get("ok"):
        r["categoria"] = clasificar(r["extraccion"])

for etiqueta, consulta in CONSULTAS_MEDICION.items():
    if resultados_medicion.get(etiqueta, {}).get("ok"):
        continue
    try:
        r = extraer(consulta)
        resultados_medicion[etiqueta] = {"consulta": consulta, "extraccion": r,
                                          "categoria": clasificar(r), "ok": True}
    except Exception as e:
        resultados_medicion[etiqueta] = {"consulta": consulta, "error": str(e), "ok": False}
    guardar_checkpoint(RUTA_MEDICION, resultados_medicion)

conteo = Counter(r["categoria"] for r in resultados_medicion.values() if r["ok"])
total = sum(conteo.values())
print("Desglose por categoria (4 causas separadas):")
for cat in ["exito", "fallback_cobertura", "fallback_fuera_tema", "fallback_ambiguedad"]:
    n = conteo.get(cat, 0)
    print(f"  {cat}: {n}/{total} ({n/total*100:.0f}%)")
print(f"\nResumen agregado WBS 4.2.3 -- exito: {conteo.get('exito',0)}/{total} | "
      f"fallback total: {total - conteo.get('exito',0)}/{total}")

banda_vigilancia = [(k, v["extraccion"]["confianza"]) for k, v in resultados_medicion.items()
                     if v["ok"] and 0.55 <= v["extraccion"]["confianza"] <= 0.75]
print(f"\nConsultas con confianza en banda 0.55-0.75 (cerca del umbral): "
      f"{banda_vigilancia if banda_vigilancia else 'ninguna'}")


Desglose por categoria (4 causas separadas):
  exito: 10/20 (50%)
  fallback_cobertura: 4/20 (20%)
  fallback_fuera_tema: 1/20 (5%)
  fallback_ambiguedad: 5/20 (25%)

Resumen agregado WBS 4.2.3 -- exito: 10/20 | fallback total: 10/20

Consultas con confianza en banda 0.55-0.75 (cerca del umbral): ninguna


**Resultado:** cada uno de los 20 casos cayó exactamente donde el diseño predecía — 10/10
válidas → éxito, 5/5 fuera de alcance repartidas correctamente entre cobertura geográfica (4:
Juan Díaz, San Miguelito, Chiriquí, Colón) y fuera de tema (1: el taller mecánico), 5/5 ambiguas →
fallback por ambigüedad. Ninguna consulta cayó en la banda de vigilancia 0.55–0.75 cercana al
umbral — no hay evidencia nueva de un punto de quiebre genuino distinto al ya observado en la
calibración.

**Advertencia explícita — mismo formato que la advertencia de validación circular de 6.2.3:** este
conjunto de 20 consultas fue diseñado deliberadamente con casos extremos por categoría (zonas
reales fuera de scope, calificativos puramente subjetivos, temas ajenos a bienes raíces) — no es
una muestra de consultas reales de usuarios. El 50% de fallback y el 100% de clasificación
correcta en este ejercicio demuestran que los tres mecanismos (confianza / cobertura / tema)
funcionan tal como se diseñaron, cada uno disparando en el caso para el que fue construido — **no
son una predicción de la tasa de fallback que se observaría en producción con consultas reales**,
que probablemente tendrán una distribución de claridad menos polarizada que este conjunto de
prueba. Cualquier lectura de estas cifras como métrica de desempeño de cara al usuario final
requeriría un conjunto muestreado de tráfico real, no diseñado por el equipo.

## Verificación end-to-end del contrato v0

La medición de la Fase 2 valida la capa de extracción + clasificación, no la orquestación
completa con M1/M2. El contrato `orquestar_m3` distingue 4 combinaciones posibles (candidatos de
M1 presentes/ausentes × cobertura de M2 disponible/no disponible) en su mensaje final — las
verificamos las 4 con evidencia real, no solo 2 como en un primer intento de esta sección.

- **V1** (real, del checkpoint de medición) — M1 con candidatos, M2 con cobertura: caso de éxito
  limpio.
- **V6** (Parque Lefevre, real) — M1 con candidatos, M2 SIN cobertura.
- **V7** (Pedregal, real) — M1 SIN candidatos, M2 SIN cobertura: el caso más exigente, confirma
  que el contrato no colapsa "sin resultados" y "sin cobertura de M2" en el mismo mensaje.
- **Caso de control adicional — M1 SIN candidatos, M2 CON cobertura.** Corriendo las 10 consultas
  "válidas" del conjunto de medición con sus criterios *reales* extraídos (no inventados) contra
  el contrato, **ninguna de las 20 ejercita esta cuarta combinación** — las que devuelven 0
  candidatos (V4, V7) lo hacen justo en zonas/tipos que tampoco tienen cobertura de M2, así que
  esa rama del código quedaba sin evidencia empírica.

  **Nota metodológica — corrección aplicada:** un primer intento de cubrir esta rama construyó un
  filtro a mano (`habitaciones_min=10` en San Francisco) para forzar el resultado, en vez de
  correr una consulta real por el pipeline completo. Ese caso fue **descartado explícitamente**
  por no ser una verificación válida — probaba que el código *podía* producir cierto texto bajo un
  input arbitrario, no que el sistema completo (extracción LLM incluida) se comportara
  correctamente ante una consulta real. Lo reemplazamos por una consulta plausible que un usuario
  real escribiría (`"Apartamento en Costa del Este por menos de $50,000"` — precio bajo poco
  realista para una zona premium, pero no un filtro absurdo), verificada primero contra el
  catálogo real (confirmamos 0 candidatos antes de usarla, no lo asumimos) y corrida por el
  pipeline completo: extracción LLM → contrato v0 → respuesta final.

In [7]:
# Verificamos primero contra el catalogo real que la consulta de control produce 0 candidatos,
# antes de usarla -- no lo asumimos.
# Aplicamos EXACTAMENTE el mismo filtro que m1_buscar_matches (incluyendo la exclusion de
# precio_no_evaluable -- 4 filas de Costa del Este tienen price_usd=1 como placeholder de
# "precio_bajo_umbral", no un precio real, y las colarian si no se excluyen aqui tambien).
_verificacion_catalogo = df_catalogo[
    (df_catalogo["corregimiento"] != "zona_no_determinada")
    & (df_catalogo["precio_no_evaluable"] == False)
    & (df_catalogo["corregimiento"] == "Costa del Este")
    & (df_catalogo["tipo_inmueble"] == "Apartamentos")
    & (df_catalogo["price_usd"] <= 50000)
]
assert len(_verificacion_catalogo) == 0, "La consulta de control ya no produce 0 candidatos -- revisar"
_costa_del_este_evaluable = df_catalogo[
    (df_catalogo["precio_no_evaluable"] == False)
    & (df_catalogo["corregimiento"] == "Costa del Este")
    & (df_catalogo["tipo_inmueble"] == "Apartamentos")
]
print(f"Verificacion contra catalogo real: {len(_verificacion_catalogo)} candidatos "
      f"(rango real de precio evaluable en Costa del Este/Apartamentos: "
      f"${_costa_del_este_evaluable['price_usd'].min():,.0f} - "
      f"${_costa_del_este_evaluable['price_usd'].max():,.0f})")

RUTA_CASO_CONTROL = RUTA_MODELOS / "caso_control_sin_candidatos_con_cobertura_6_2_7.pkl"
CONSULTA_CONTROL = "Apartamento en Costa del Este por menos de $50,000"

caso_control = cargar_checkpoint(RUTA_CASO_CONTROL)
if not caso_control:
    extraccion_control = extraer(CONSULTA_CONTROL)
    caso_control = {"consulta": CONSULTA_CONTROL, "extraccion": extraccion_control}
    guardar_checkpoint(RUTA_CASO_CONTROL, caso_control)

print(f"\nExtraccion real (LLM) para el caso de control: {caso_control['extraccion']}")

# --- Los 3 casos ya confirmados usan los criterios REALES extraidos en la Fase 2 (checkpoint),
# no criterios reescritos a mano.
_campos = ["zona", "tipo_inmueble", "precio_min", "precio_max", "habitaciones_min", "banos_min"]
casos_verificacion = {
    "V1 (con cand. x con cobertura)": {c: resultados_medicion["V1"]["extraccion"][c] for c in _campos},
    "V6 (con cand. x SIN cobertura)": {c: resultados_medicion["V6"]["extraccion"][c] for c in _campos},
    "V7 (SIN cand. x SIN cobertura)": {c: resultados_medicion["V7"]["extraccion"][c] for c in _campos},
    "CONTROL (SIN cand. x con cobertura)": {c: caso_control["extraccion"][c] for c in _campos},
}

for etiqueta, criterios in casos_verificacion.items():
    r = orquestar_m3(criterios)
    print(f"\n--- {etiqueta} ---")
    print(f"  criterios: {criterios}")
    print(f"  M1: n_total_filtrado={r['m1']['n_total_filtrado']}")
    print(f"  M2: cobertura_insuficiente={r['m2']['cobertura_insuficiente']}"
          + (f" ({r['m2']['motivo']})" if r['m2']['cobertura_insuficiente'] else ""))
    print(f"  Respuesta: {r['respuesta_final']}")


Verificacion contra catalogo real: 0 candidatos (rango real de precio evaluable en Costa del Este/Apartamentos: $140,000 - $3,365,149)

Extraccion real (LLM) para el caso de control: {'es_consulta_inmobiliaria': True, 'zona': 'Costa del Este', 'zona_mencion_texto': 'Costa del Este', 'tipo_inmueble': 'Apartamento', 'precio_min': None, 'precio_max': 50000, 'habitaciones_min': None, 'banos_min': None, 'caracteristicas_cualitativas': [], 'confianza': 0.95, 'razon_confianza': 'Entidades claras de zona, tipo de inmueble y precio máximo sin ambigüedades.'}

--- V1 (con cand. x con cobertura) ---
  criterios: {'zona': 'El Cangrejo', 'tipo_inmueble': 'Apartamento', 'precio_min': None, 'precio_max': 150000, 'habitaciones_min': 1, 'banos_min': None}
  M1: n_total_filtrado=2
  M2: cobertura_insuficiente=False
  Respuesta: Encontramos 2 propiedades que coinciden con tu busqueda, con semaforo de precio incluido.

--- V6 (con cand. x SIN cobertura) ---
  criterios: {'zona': 'Parque Lefevre', 'tipo_in

**Las 4 combinaciones quedan confirmadas con evidencia real, no inventada:**

1. **Con candidatos × con cobertura (V1):** respuesta positiva simple, semáforo incluido.
2. **Con candidatos × SIN cobertura (V6, Parque Lefevre):** muestra los candidatos reales
   encontrados y explica, con el motivo estructural exacto, por qué no hay semáforo de precio.
3. **SIN candidatos × SIN cobertura (V7, Pedregal):** confirmado en las 4 condiciones de éxito —
   (a) extracción correcta (`zona=Pedregal, tipo_inmueble=Apartamento, habitaciones_min=3`); (b)
   `confianza=1.00 > 0.65`; (c) M1 filtra el catálogo real y devuelve 0 candidatos (dato real: las
   3 filas de Pedregal en el catálogo son 2 apartamentos de 2 habitaciones y 1 terreno, ninguna
   cumple 3+ habitaciones), M2 detecta la falta estructural de cobertura antes de intentar
   predecir; (d) la respuesta es honesta sobre ambos hechos por separado.
4. **SIN candidatos × con cobertura (caso de control, Costa del Este < \$50,000):** la única
   combinación que ningún caso del conjunto de 20 ejercitaba naturalmente. Confirmada con una
   consulta real corrida por el pipeline completo (extracción LLM → contrato → respuesta), no con
   el filtro artificial descartado. El texto final reconoce 0 candidatos de inmediato y ofrece una
   sugerencia accionable, sin mencionar limitaciones de M2 que no aplican aquí (Costa del Este sí
   tiene cobertura del modelo).

**Dos hallazgos reales encontrados durante esta verificación, no cosméticos:**

- El primer borrador del ensamblado de respuesta solo distinguía "con/sin cobertura de M2", sin
  rama separada para "M1 sin candidatos". Para V7 esto producía un mensaje engañoso ("te
  mostramos los resultados... sin ese dato" cuando no había ningún resultado). Corregido separando
  las 4 combinaciones antes de dar V7 por confirmado.
- La primera verificación de la combinación 4 usó un filtro construido a mano
  (`habitaciones_min=10` en San Francisco) en vez de una consulta real corrida por el pipeline
  completo — un atajo que probaba el código, no el sistema. Descartado explícitamente y
  reemplazado por la consulta de control documentada arriba, verificada contra el catálogo real
  antes de usarla y corrida por el mismo pipeline que las demás (extracción LLM incluida).

## Conclusiones

Cerramos Feature 6.2.7 con un flujo de orquestación validado en sus tres componentes: extracción
de intención con campo de confianza explícito, dos chequeos deterministicos adicionales
(cobertura geográfica y relevancia de tema) que el score de confianza por sí solo no puede
resolver, y un contrato de función v0 que orquesta M1/M2 con respuestas honestas sobre las
limitaciones estructurales ya documentadas del catálogo (Pedregal y Parque Lefevre excluidos de
KNN/RF/KMeans por volumen, Acta 1.2 §5.3). Las 4 combinaciones posibles de la respuesta final
(candidatos de M1 presentes/ausentes × cobertura de M2 disponible/no disponible) quedaron
verificadas con evidencia real del pipeline completo, incluyendo la que ningún caso del conjunto
de 20 ejercitaba naturalmente.

**Tres decisiones quedan documentadas como resultado de este notebook, no como supuestos de
entrada:**

1. **Modelo LLM:** `gemini-3.5-flash` (GA, identificador fijo) — verificado contra la API real con
   el mismo criterio de lote de 25 llamadas que estableció 6.2.6, con resultado de 100% de éxito
   en esta corrida (vs. 100% de fallo sostenido en 6.2.6). La disponibilidad de modelos Gemini es
   una condición del momento de ejecución, no una garantía permanente.
2. **Umbral de confianza 0.65**, con sesgo conservador documentado y marcado explícitamente como
   calibración provisional — no optimizada, dado el tamaño y diseño del conjunto de calibración.
3. **Mecanismo de fallback de tres causas separables** (ambigüedad, cobertura geográfica, fuera de
   tema), no un solo score de confianza — necesario porque el score de confianza del LLM, por
   diseño, no puede distinguir "una consulta bien formada sobre una zona que no cubrimos" de "una
   consulta genuinamente vaga".

**Limitación conocida, dirección de trabajo futura (no implementada en 6.2.7):** el punto de
quiebre real del umbral de confianza permanece sin observar directamente — el conjunto de 20 de
medición final, diseñado con casos extremos por categoría, no produjo ningún caso con confianza
entre 0.55 y 0.75 que sirviera de evidencia adicional. Ajustar el umbral con más certeza
requeriría un conjunto de consultas reales de usuarios, no diseñado por el equipo con la intención
de caer en un extremo u otro — tarea de Épica 4, no de este notebook.

Con este cierre, **Feature 6.2 (23 SP) queda completa** — los 7 notebooks de análisis y
entrenamiento (EDA, Zone Health, M1, M2 KNN/RF, M2 KMeans, M2 Quality Scorer, M3 Orchestration)
están documentados con evidencia de proceso, listos como fuente de verdad de los modelos que el
backend de Épica 4 consumirá.